In [1]:
import pandas as pd
import numpy as np
from nltk.corpus import words as nltk_words
import sys
# Add the parent directory to Python's module search path
sys.path.append('..')
from src.run import run_steps
from src.steps import load_words_to_set
import concurrent.futures
from functools import partial
import os
from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TextGenerationPipeline,
)
import torch
from src.steps import (
    extract_sentiment_label,
    score_from_label,
)

c:\Users\HP\Desktop\tiktok-analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_excel('C:/Users/HP/Desktop/tiktok-analysis/dataset_tiktok-comments-scraper_2026-04-22_19-27-52-195.xlsx')

In [3]:
df["commentText"] = df["commentText"].replace(r'^\s*$', np.nan, regex=True)
df = df.dropna(subset=["commentText"])

In [4]:
comments_str = df["commentText"].str.cat(sep=' ')

In [5]:
words_dict_set: dict[str, set[str]] = {
        "English": set(nltk_words.words()),
        "Luganda": load_words_to_set(
            file_path="C:/Users/HP/Desktop/tiktok-analysis/words/processed/luganda_words.txt"
        ),
        "Swahili": load_words_to_set(
            file_path="C:/Users/HP/Desktop/tiktok-analysis/words/processed/swahili_words.txt"
        ),
    }

In [6]:
words_dict_list: dict[str, list[str]] = {
        "English": list(words_dict_set["English"]),
        "Luganda": list(words_dict_set["Luganda"]),
        "Swahili": list(words_dict_set["Swahili"])
    }

In [10]:
run_steps(initial_word="nyabo", words_dict_set=words_dict_set, words_dict_list = words_dict_list)

('Luganda', ('nnyabo', 90.9090909090909, 166))

I honestly think fuzzy match can handle at least 90% of misspellings or unknown words from the lexicon lookups... So we may not need the n-grams classifier after all... but let's run it through all the comment words and then we get the percentage that after passing "run_steps" on, returns "None"... Only if a small percentage is caught will I consider the classifier...

In [11]:
all_words = comments_str.split()
unique_words = list(set(all_words))

# Prepare the target function
# The executor's map() function expects a target function that takes exactly ONE argument (the word).
# We use partial() to "freeze" your dictionary arguments into the function ahead of time... so we will be inputting just one.
run_steps_partial = partial(
    run_steps,
    words_dict_set=words_dict_set,
    words_dict_list=words_dict_list
)

max_cores = os.cpu_count() or 12
print(f"Processing {len(unique_words)} across {max_cores} cores...")

# 3. Process in parallel
# ProcessPoolExecutor creates separate Python processes to bypass Python's single-core limitation
with concurrent.futures.ProcessPoolExecutor(max_workers=max_cores) as executor:
    # executor.map takes our modified function and applies it to every unique word
    results_list = list(executor.map(run_steps_partial, unique_words))

# Create the O(1) lookup
word_results_lookup = dict(zip(unique_words, results_list))


percent_list = []
count = 0

for word in all_words:
    result = word_results_lookup[word]
    percent_list.append(result)
    if result == "None":
        count = count + 1

Processing 3233 across 12 cores...


In [12]:
none_percent = (count/len(percent_list))*100
print("Percentage of 'None' results: ",none_percent)

Percentage of 'None' results:  10.377430656487615


Definitely not worth the trouble!

Now let's do the proper runthorugh for ALL words. Execute the CPU bound tasks

In [7]:
all_words = comments_str.split()

# Prepare the target function
# The executor's map() function expects a target function that takes exactly ONE argument (the word).
# We use partial() to "freeze" your dictionary arguments into the function ahead of time... so we will be inputting just one.
run_steps_partial = partial(
    run_steps,
    words_dict_set=words_dict_set,
    words_dict_list=words_dict_list
)

max_cores = os.cpu_count() or 12
print(f"Processing {len(all_words)} across {max_cores} cores...")

# 3. Process in parallel
# ProcessPoolExecutor creates separate Python processes to bypass Python's single-core limitation
with concurrent.futures.ProcessPoolExecutor(max_workers=max_cores) as executor:
    # executor.map takes our modified function and applies it to every word
    results_list = list(executor.map(run_steps_partial, all_words))

# Create the O(1) lookup
#word_results_lookup = dict(zip(all_words, results_list))

No GPU detected. Running everything on CPU across 12 cores.
Processing 3233 unique words across 12 cores (chunksize=67)


Execute the GPU bound tasks

In [23]:
generation_pipelines: dict = {"English": None, "Luganda": None, "Swahili": None}

model_g_path = "C:/Users/HP/ganda-gemma-1b"
model_s_path = "C:/Users/HP/swahili-gemma-1b"

#quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# config to share across models
load_args = {
 #   "quantization_config": quantization_config,
    "low_cpu_mem_usage": True,
}

In [ ]:
pending_luganda_words=[]
pending_swahili_words=[]

In [9]:
pending_luganda_words = [result for result in results_list if result["status"] == "pending_gpu" and result["checked_lang"] == "Luganda"]
pending_swahili_words = [result for result in results_list if result["status"] == "pending_gpu" and result["checked_lang"] == "Swahili"]

In [29]:
print(pending_luganda_words)

[{'original_text': 'naye', 'normalized_word_text': 'naye', 'dict_word': 'Luganda', 'dict_lang': 'Luganda', 'fuzzy_word': '', 'fuzzy_lang': '', 'fuzzy_confidence': 0.0, 'checked_text': 'Luganda', 'checked_lang': 'Luganda', 'lemmatized_text': '', 'polarity_score': 0.0, 'polarity_label': '', 'status': 'ok', 'error_message': ''}, {'original_text': 'naye', 'normalized_word_text': 'naye', 'dict_word': 'Luganda', 'dict_lang': 'Luganda', 'fuzzy_word': '', 'fuzzy_lang': '', 'fuzzy_confidence': 0.0, 'checked_text': 'Luganda', 'checked_lang': 'Luganda', 'lemmatized_text': '', 'polarity_score': 0.0, 'polarity_label': '', 'status': 'ok', 'error_message': ''}, {'original_text': 'wano', 'normalized_word_text': 'wano', 'dict_word': 'Luganda', 'dict_lang': 'Luganda', 'fuzzy_word': '', 'fuzzy_lang': '', 'fuzzy_confidence': 0.0, 'checked_text': 'Luganda', 'checked_lang': 'Luganda', 'lemmatized_text': '', 'polarity_score': 0.0, 'polarity_label': '', 'status': 'pending_gpu', 'error_message': ''}, {'origina

In [ ]:
# iterate through pending with formatted prompt

In [ ]:
for lang_name, pending_words in [("Luganda", pending_luganda_words), ("Swahili", pending_swahili_words)]:
    if not pending_words:
        continue

    # Load pipeline for the current language only inside the loop
    if generation_pipelines[lang_name] is None:
        model_path = model_g_path if lang_name == "Luganda" else model_s_path
        model = AutoModelForCausalLM.from_pretrained(model_path, **load_args)
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        generation_pipelines[lang_name] = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            device=0 if torch.cuda.is_available() else -1,
        )

    gen_pipe = generation_pipelines[lang_name]

    # Build prompts for this language batch
    sentiment_prompts = [
        f"Classify the sentiment of this {lang_name} text. Answer with one word only: positive, neutral or negative. Text: '{item['checked_text']}'. Answer in {lang_name}."
        for item in pending_words
    ]

    # Run the batch through the pipeline (returns list-of-lists/dicts)
    outputs = gen_pipe(
        sentiment_prompts,
        max_new_tokens=5,
        do_sample=False,
        temperature=0.1,
        return_full_text=False,
    )

    # Map outputs back into the pending dicts
    for item, out in zip(pending_words, outputs):
        gen_text = out[0]["generated_text"].strip() if isinstance(out, list) else out.get("generated_text", "").strip()
        #label = extract_sentiment_label(gen_text)
        #score = score_from_label(label)
        item["polarity_label"] = gen_text
        #item["polarity_score"] = score
        item["status"] = "ok"

In [ ]:
# Smoke test the GPU path on a tiny sample before running the full batch.
smoke_luganda = pending_luganda_words[:2]
smoke_swahili = pending_swahili_words[:2]

smoke_results = []

for lang_name, pending_words in [("Luganda", smoke_luganda), ("Swahili", smoke_swahili)]:
    if not pending_words:
        continue

    if generation_pipelines[lang_name] is None:
        model_path = model_g_path if lang_name == "Luganda" else model_s_path
        model = AutoModelForCausalLM.from_pretrained(model_path, **load_args)
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        generation_pipelines[lang_name] = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            device=0 if torch.cuda.is_available() else -1,
        )

    gen_pipe = generation_pipelines[lang_name]

    smoke_prompts = [
        f"Classify the sentiment of this {lang_name} text. Answer with one word only: positive, neutral or negative. Text: '{item['checked_text']}'. Answer in {lang_name}"
        for item in pending_words
    ]

    outputs = gen_pipe(
        smoke_prompts,
        max_new_tokens=5,
        do_sample=False,
        temperature=0.1,
        return_full_text=False,
    )

    for item, out in zip(pending_words, outputs):
        raw_text = out[0]["generated_text"].strip() if isinstance(out, list) else out.get("generated_text", "").strip()
        #label = extract_sentiment_label(raw_text)
        #score = score_from_label(label)
        item["polarity_label"] = raw_text
        #item["polarity_score"] = score
        item["status"] = "ok"
        smoke_results.append({
            "language": lang_name,
            "text": item["checked_text"],
            "raw_output": raw_text,
            "label": raw_text,
            #"score": score,
        })

pd.DataFrame(smoke_results)

[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,language,text,raw_output,label
0,Luganda,Luganda,,
1,Luganda,Luganda,,
2,Swahili,adawa,,
3,Swahili,boosa,,


Put it all in a df

In [25]:
# put in a dataframe, put together repeating words and add a column for term frequency
words = pd.DataFrame(data=results_list, columns=["corrected_text", "checked_lang", "polarity_score", "polarity_label"])

words.rename(columns={"corrected_text": "Word", "checked_lang": "Language", "polarity_score": "Polarity Score", "polarity_label": "Polarity Label"}, inplace=True)

# to add term frequency to df
words = (
    words.groupby(["Word", "Language", "Polarity Score", "Polarity Label"]).size().reset_index(name="Term Frequency")
)

In [26]:
words.head()

,Word,Language,Polarity Score,Polarity Label,Term Frequency
